# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring the FAIR\u2072 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and field `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)
print("Available Record Sets (listed by their @id):\n")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '<no name>')})")

if record_sets:
    print("\nField overviews for each Record Set:")
    for rs in record_sets:
        print(f"\nRecord Set @id: {rs['@id']} (name: {rs.get('name', '<no name>')})")
        fields = rs.get('field', []) if isinstance(rs.get('field', []), list) else [rs.get('field', [])]
        if fields and fields[0]:
            for field in fields:
                print(f"  - Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, type: {field.get('dataType', '<unknown>')}")
        else:
            print("  (No field objects found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All accesses use the record set and field `@id`s identified above.

In [ ]:
# Collect record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Use the @id to access records
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
        else:
            print(f"No records found for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Show columns from first available DataFrame
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nFields (@id) in DataFrame for {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No DataFrames were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, and group/categorize data. All operations are referenced by their `@id` field names.

In [ ]:
# Select the first available data for analysis
if dataframes:
    record_set_id = first_rs_id  # Use the first loaded record set
    df = dataframes[record_set_id]
    print(f"Working with DataFrame for record set @id: {record_set_id}\nAvailable columns (field @id):\n{df.columns.tolist()}")
    
    # Try to find a likely numeric field
    # If a field contains only numbers, is 'abundance' or 'MW' or similar, use it
    inferred_numeric = None
    numeric_candidates = [c for c in df.columns if 'abundance' in c or 'MW' in c or 'count' in c or 'peptide' in c or 'number' in c or df[c].dtype.kind in ('i', 'f')]
    for c in numeric_candidates:
        try:
            nums = pd.to_numeric(df[c], errors='coerce')
            if nums.notna().sum()>0:
                inferred_numeric = c
                break
        except Exception:
            continue
    if inferred_numeric is None:
        # Default to the first column if all else fails
        inferred_numeric = df.columns[0]
    print(f"Selected field for numeric analysis (by @id): {inferred_numeric}")
    
    # Filter for values above mean (will adapt threshold after checking values)
    nums = pd.to_numeric(df[inferred_numeric], errors='coerce')
    threshold = nums.mean() if nums.notna().sum() else 0
    filtered_df = df[nums > threshold].copy()
    print(f"Filtered records with {inferred_numeric} > {threshold} (mean): Found {len(filtered_df)} records.")
    display(filtered_df.head())

    # Normalize numeric field
    if filtered_df.shape[0]>0:
        filtered_df[f"{inferred_numeric}_normalized"] = (pd.to_numeric(filtered_df[inferred_numeric], errors='coerce') - nums.mean()) / nums.std()
        print(f"Normalized {inferred_numeric} for filtered records:")
        display(filtered_df[[inferred_numeric, f"{inferred_numeric}_normalized"]].head())
    
    # Try to group by a categorical field
    group_fields = [c for c in df.columns if c!=inferred_numeric and (df[c].dtype=='O' or 'sample' in c or 'group' in c or 'condition' in c)]
    if group_fields:
        group_field = group_fields[0]
        print(f"\nGrouping by field (by @id): {group_field}")
        grouped_df = filtered_df.groupby(group_field)[inferred_numeric].mean().to_frame()
        display(grouped_df.head())
    else:
        group_field = None
        print("No obvious group/categorical field for grouping.")
else:
    print("No DataFrames to analyze.")

## 5. Visualization
Visualize value distribution for the selected numeric field and grouped means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Plot the histogram of the selected numeric field
    plt.figure(figsize=(7,3))
    nums = pd.to_numeric(df[inferred_numeric], errors='coerce')
    sns.histplot(nums.dropna(), bins=25)
    plt.title(f"Distribution of {inferred_numeric}")
    plt.xlabel(inferred_numeric)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    # If grouping done, plot group means
    if group_field is not None:
        grouped = filtered_df.groupby(group_field)[inferred_numeric].mean()
        grouped.sort_values(ascending=False).plot(kind='bar', figsize=(8,3))
        plt.title(f"Mean {inferred_numeric} by {group_field}")
        plt.ylabel(f"Mean {inferred_numeric}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No DataFrames loaded for visualization.")

## 6. Conclusion
This notebook guided you through exploring the FAIR\u00b2 mass spectrometry dataset using `mlcroissant`, including metadata inspection, record extraction by `@id`, exploratory data analysis, and basic visualizations.

Key takeaways:
- Record sets, fields, and columns are referenced using their `@id` in all steps for reproducible analyses.
- The `mlcroissant` library enables easier and more reliable FAIR-aligned biomedical data loading.
- You can extend the EDA and data wrangling sections to suit downstream analysis.

See [mlcroissant documentation](https://github.com/mlcommons/croissant) for further details.